In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import warnings
import time

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
warnings.filterwarnings('ignore', category=Warning)

print("=== MODELO ULTRA_PLUS_V2 — GLOBAL + ESPECIALISTAS + TUNING AUTOMÁTICO ===\n")

# ==================================================
#  FUNÇÕES DE PRÉ-PROCESSAMENTO (IGUAIS ÀS TUAS)
# ==================================================

def formatar_celula(series_coluna):
    s = series_coluna.astype(str).replace('NULL', pd.NA)
    s = s.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    s = s.str.upper()
    s = s.str.replace(r'[^A-Z0-9]+', '_', regex=True)
    s = s.str.strip('_')
    s = s.replace('', pd.NA)
    return s

def preprocessar_dados(df, colunas_scaler_treinadas=None, scaler=None):
    if 'RowId' not in df.columns and 'AVERAGE_SPEED_DIFF' not in df.columns:
        df_final_row_ids = np.arange(1, len(df) + 1)
    else:
        df_final_row_ids = None

    cols_to_transform = ['AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']
    for col in cols_to_transform:
        if col in df.columns:
            df[col] = formatar_celula(df[col])

    cols_to_drop_base = ['city_name', 'AVERAGE_RAIN', 'AVERAGE_PRECIPITATION', 'record_date']

    # Features de data / hora
    try:
        df['record_date_dt'] = pd.to_datetime(df['record_date'], format='mixed', dayfirst=True)
        df['Hora_sin'] = np.sin(2 * np.pi * df['record_date_dt'].dt.hour / 24)
        df['Hora_cos'] = np.cos(2 * np.pi * df['record_date_dt'].dt.hour / 24)
        df['Mes_sin'] = np.sin(2 * np.pi * df['record_date_dt'].dt.month / 12)
        df['Mes_cos'] = np.cos(2 * np.pi * df['record_date_dt'].dt.month / 12)
        df['DIA_SEMANA'] = df['record_date_dt'].dt.dayofweek
        df['IS_WEEKEND'] = df['DIA_SEMANA'].isin([5, 6]).astype(int)
        rush_hours = [7, 8, 9, 17, 18, 19]
        df['IS_RUSH_HOUR'] = df['record_date_dt'].dt.hour.isin(rush_hours).astype(int)
        df = df.drop(columns=['record_date_dt'])
    except KeyError:
        pass

    cols_existentes_drop = [col for col in cols_to_drop_base if col in df.columns]
    df = df.drop(columns=cols_existentes_drop)

    # Tratamento da cloudiness
    if 'AVERAGE_CLOUDINESS' in df.columns:
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('NAN', np.nan)
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].fillna('NONE')

    # One-hot
    cols_to_onehot = ['LUMINOSITY', 'AVERAGE_CLOUDINESS', 'DIA_SEMANA']
    for col in cols_to_onehot:
        if col in df.columns:
            prefix = col[:3].upper()
            if col == 'DIA_SEMANA':
                prefix = 'DAY'
            dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int, sparse=False)
            df = pd.concat([df, dummies], axis=1)
            df = df.drop(col, axis=1)

    # Normalização
    cols_to_normalize = [
        'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME',
        'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY',
        'AVERAGE_WIND_SPEED', 'IS_WEEKEND', 'IS_RUSH_HOUR',
        'Hora_sin', 'Hora_cos', 'Mes_sin', 'Mes_cos'
    ]
    cols_existentes_normalize = [col for col in cols_to_normalize if col in df.columns]

    if scaler is None:
        scaler = MinMaxScaler()
        if cols_existentes_normalize:
            df[cols_existentes_normalize] = scaler.fit_transform(df[cols_existentes_normalize])
        return df, scaler, cols_existentes_normalize, df_final_row_ids
    else:
        cols_para_scaler_teste = [col for col in colunas_scaler_treinadas if col in df.columns]
        if cols_para_scaler_teste:
            df[cols_para_scaler_teste] = scaler.transform(df[cols_para_scaler_teste])
        return df, None, None, df_final_row_ids


# ==================================================
#  CARREGAR DADOS
# ==================================================

print("\n--- [Passo 1/7] A carregar dados de Treino e Teste... ---")
df_train = pd.read_csv("training_data.csv", delimiter=",", encoding="latin-1")
df_test = pd.read_csv("test_data.csv", delimiter=",", encoding="latin-1")

y_train_raw = df_train.pop('AVERAGE_SPEED_DIFF')
test_row_ids = np.arange(1, len(df_test) + 1)

print("--- [Passo 2/7] A pré-processar dados de treino... ---")
X_train, scaler, colunas_scaler, _ = preprocessar_dados(df_train)

le = LabelEncoder()
y_train_formatado = formatar_celula(y_train_raw).replace('NAN', 'NONE').fillna('NONE')
y_train_encoded = le.fit_transform(y_train_formatado)

print("--- [Passo 3/7] A pré-processar dados de TESTE... ---")
X_test, _, _, _ = preprocessar_dados(df_test, colunas_scaler_treinadas=colunas_scaler, scaler=scaler)

print("--- [Passo 4/7] A alinhar colunas de Treino e Teste... ---")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train = X_train.reindex(columns=X_test.columns, fill_value=0)

classes = le.classes_  # ['HIGH','LOW','MEDIUM','NONE','VERY_HIGH']

# ==================================================
#  MODELO GLOBAL (STACKING MULTICLASS)
# ==================================================

print("\n--- [Passo 5/7] Modelo Global (Stacking XGB + LGBM) ---")

estimators = [
    ('xgb', XGBClassifier(
        learning_rate=0.08, max_depth=5, n_estimators=120,
        subsample=0.7, colsample_bytree=0.9,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=42, n_jobs=-1
    )),
    ('lgbm', LGBMClassifier(
        objective='multiclass',
        num_class=len(classes),
        boosting_type='gbdt',
        learning_rate=0.04,
        num_leaves=31,
        n_estimators=350,
        min_data_in_leaf=40,
        feature_fraction=0.75,
        bagging_fraction=0.75,
        bagging_freq=3,
        lambda_l1=1.5,
        lambda_l2=1.5,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
]

meta_model = LogisticRegression(
    random_state=42,
    n_jobs=-1,
    max_iter=1000,
    multi_class='auto'
)

stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

print("A calcular o Cross-Validation do modelo global...")
start_time = time.time()
scores = cross_val_score(
    stack_clf, X_train, y_train_encoded,
    cv=5, scoring='accuracy', n_jobs=1, verbose=1
)
end_time = time.time()
print(f"Accuracy CV (modelo global): {np.mean(scores):.5f}")
print(f"Tempo CV: {end_time - start_time:.1f} segundos")

print("\nA treinar o modelo global final em todo o treino...")
stack_clf.fit(X_train, y_train_encoded)

proba_global_train = stack_clf.predict_proba(X_train)
df_global_train = pd.DataFrame(proba_global_train, columns=classes)
pred_train_global = df_global_train.idxmax(axis=1)
acc_g = accuracy_score(y_train_formatado, pred_train_global)
print(f"🔵 Accuracy Global (in-sample): {acc_g:.4f}")

proba_global_test = stack_clf.predict_proba(X_test)
df_global_test = pd.DataFrame(proba_global_test, columns=classes)

# ==================================================
#  SISTEMA DE ESPECIALISTAS — COM OUT-OF-FOLD
# ==================================================

print("\n--- [Passo 6/7] Especialistas One-vs-All (com OOF) ---")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

especialistas = {}
scores_f1 = {}
df_spec_train_raw = pd.DataFrame(index=X_train.index, columns=classes, dtype=float)
df_spec_test_raw = pd.DataFrame(index=X_test.index, columns=classes, dtype=float)

for classe in classes:
    print(f"\n🔍 Treinar especialista para: {classe}")

    y_bin = (y_train_formatado == classe).astype(int).values
    pos = int((y_bin == 1).sum())
    neg = int((y_bin == 0).sum())

    weight = (neg / pos) if pos > 0 else 1
    weight = min(max(weight, 1.0), 10.0)

    # Modelo do especialista
    def criar_especialista():
        return StackingClassifier(
            estimators=[
                ("xgb", XGBClassifier(
                    learning_rate=0.05,
                    max_depth=4,
                    n_estimators=130,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    eval_metric='logloss',
                    random_state=42,
                    scale_pos_weight=weight,
                    n_jobs=-1
                )),
                ("lgbm", LGBMClassifier(
                    objective="binary",
                    n_estimators=170,
                    num_leaves=24,
                    min_data_in_leaf=35,
                    learning_rate=0.03,
                    feature_fraction=0.8,
                    bagging_fraction=0.8,
                    bagging_freq=3,
                    lambda_l1=1.0,
                    lambda_l2=1.0,
                    class_weight={0: 1.0, 1: weight},
                    random_state=42,
                    n_jobs=-1
                ))
            ],
            final_estimator=LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=42,
                n_jobs=-1
            ),
            cv=3,
            n_jobs=-1
        )

    oof_proba = np.zeros(len(X_train))
    oof_pred = np.zeros(len(X_train))

    # OOF training
    for train_idx, val_idx in skf.split(X_train, y_bin):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr = y_bin[train_idx]

        esp_fold = criar_especialista()
        esp_fold.fit(X_tr, y_tr)

        p_val = esp_fold.predict_proba(X_val)[:, 1]
        oof_proba[val_idx] = p_val
        oof_pred[val_idx] = (p_val >= 0.5).astype(int)

    f1 = f1_score(y_bin, oof_pred)
    scores_f1[classe] = max(f1, 0.30)
    df_spec_train_raw[classe] = oof_proba * scores_f1[classe]

    print(f"   → F1 OOF especialista {classe}: {f1:.4f}  (peso aplicado: {scores_f1[classe]:.3f})")

    # Treinar especialista final em TODO o treino para depois prever TESTE
    especialista_full = criar_especialista()
    especialista_full.fit(X_train, y_bin)
    especialistas[classe] = especialista_full

    p_test = especialista_full.predict_proba(X_test)[:, 1]
    df_spec_test_raw[classe] = p_test * scores_f1[classe]

# Normalização das probabilidades dos especialistas
def normalizar_especialistas(df_raw):
    row_sums = df_raw.sum(axis=1).replace(0, np.nan)
    return df_raw.div(row_sums, axis=0).fillna(0.0)

df_spec_train_norm = normalizar_especialistas(df_spec_train_raw)
df_spec_test_norm = normalizar_especialistas(df_spec_test_raw)

# ==================================================
#  TUNING AUTOMÁTICO: alpha, threshold, boost_VH
# ==================================================

print("\n--- [Passo 7/7] Tuning automático ULTRA_PLUS_V2 ---")

alpha_grid = np.linspace(0.20, 0.50, 7)        # peso dos especialistas
thr_grid = np.linspace(0.45, 0.85, 9)         # confiança do global
boost_grid = [0.00, 0.03, 0.05, 0.07]         # extra para VERY_HIGH

maxprob_global_train = df_global_train.max(axis=1)

melhor_combo = None
melhor_acc = -1.0

for alpha in alpha_grid:
    # blend base treino
    df_blend_base = (1 - alpha) * df_global_train + alpha * df_spec_train_norm

    for boost in boost_grid:
        df_blend = df_blend_base.copy()

        if "VERY_HIGH" in df_blend.columns and boost > 0:
            # só boost onde o especialista acredita mais em VERY_HIGH
            mask_vh = df_spec_train_raw["VERY_HIGH"] > 0.6
            df_blend.loc[mask_vh, "VERY_HIGH"] += boost

        df_blend = df_blend.div(df_blend.sum(axis=1), axis=0)
        pred_ultra_train = df_blend.idxmax(axis=1)

        for thr in thr_grid:
            pred_mix = pred_ultra_train.copy()
            mask_conf = maxprob_global_train >= thr
            pred_mix[mask_conf] = pred_train_global[mask_conf]

            acc_mix = accuracy_score(y_train_formatado, pred_mix)

            if acc_mix > melhor_acc:
                melhor_acc = acc_mix
                melhor_combo = (alpha, thr, boost)

alpha_best, thr_best, boost_best = melhor_combo
print(f"\n✅ Melhor combinação encontrada:")
print(f"   alpha (peso especialistas) = {alpha_best:.3f}")
print(f"   threshold confiança global = {thr_best:.3f}")
print(f"   boost VERY_HIGH            = {boost_best:.3f}")
print(f"⚡ Accuracy ULTRA+ OOF (train): {melhor_acc:.4f}")

# ==================================================
#  APLICAR COMBINAÇÃO ÓTIMA AO TESTE
# ==================================================

# BLEND no TESTE
df_blend_base_test = (1 - alpha_best) * df_global_test + alpha_best * df_spec_test_norm
df_blend_test = df_blend_base_test.copy()

if "VERY_HIGH" in df_blend_test.columns and boost_best > 0:
    mask_vh_test = df_spec_test_raw["VERY_HIGH"] > 0.6
    df_blend_test.loc[mask_vh_test, "VERY_HIGH"] += boost_best

df_blend_test = df_blend_test.div(df_blend_test.sum(axis=1), axis=0)

pred_ultra_test = df_blend_test.idxmax(axis=1)
maxprob_global_test = df_global_test.max(axis=1)
pred_global_test = df_global_test.idxmax(axis=1)

pred_final_categorias = pred_ultra_test.copy()
mask_conf_test = maxprob_global_test >= thr_best
pred_final_categorias[mask_conf_test] = pred_global_test[mask_conf_test]

# Converter para o formato requerido (Title Case)
pred_final = pred_final_categorias.str.title()

submission_df = pd.DataFrame({
    "RowId": test_row_ids,
    "Speed_Diff": pred_final
})
submission_df.to_csv("submission_especialistas_ULTRA_PLUS_V2.csv", index=False)

print("\n============= ULTRA_PLUS_V2 FINALIZADO =============")
print(" Submission gerada -> submission_especialistas_ULTRA_PLUS_V2.csv")
print("=====================================================\n")


=== MODELO ULTRA_PLUS_V2 — GLOBAL + ESPECIALISTAS + TUNING AUTOMÁTICO ===


--- [Passo 1/7] A carregar dados de Treino e Teste... ---
--- [Passo 2/7] A pré-processar dados de treino... ---
--- [Passo 3/7] A pré-processar dados de TESTE... ---
--- [Passo 4/7] A alinhar colunas de Treino e Teste... ---

--- [Passo 5/7] Modelo Global (Stacking XGB + LGBM) ---
A calcular o Cross-Validation do modelo global...


[20:21:26] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:21:26] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:21:26] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:21:26] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:21:26] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:22:09] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:22:09] WARNING: /Users/runner/minifo

Accuracy CV (modelo global): 0.81180
Tempo CV: 192.7 segundos

A treinar o modelo global final em todo o treino...


[20:24:32] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:24:32] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:24:32] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:24:32] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[20:24:32] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.



🔵 Accuracy Global (in-sample): 0.9225

--- [Passo 6/7] Especialistas One-vs-All (com OOF) ---

🔍 Treinar especialista para: HIGH
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=35, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=35
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current valu